In [0]:
# Set the 'dt' parameter dynamically using dbutils
#dbutils.widgets.text("dt", "2025-01-07", "Execution Date")
#dt_value = dbutils.widgets.get("dt")

dbutils.widgets.text("dt", "2025-01-11", "Execution Date")
dt_value = dbutils.widgets.get("dt")

# Validate if 'dt' is empty or null and throw an error
if not dt_value:
    raise ValueError("ERROR: The 'dt' parameter is missing or empty. Please provide a valid date in 'yyyy-MM-dd' format.")
else:
    print(f"Running query for date: {dt_value}")







In [0]:
# Overwrite the managed table on each run
spark.sql(f"""
CREATE OR REPLACE TABLE revr_bmbs_dmp_offline.recommender_output_priority (
    vin STRING,
    product_id STRING,
    product_type STRING,
    product_category STRING,
    score DOUBLE,
    loyalty_flag INT,
    tire_package_flag INT,
    maintenance_flag INT,
    ew_flag INT,
    lpo_flag INT,
    mmc_flag INT,
    mmc_coupon_flag INT,
    mmc_priority_rank INT,
    adjusted_score DOUBLE,
    dt STRING
)
""")


In [0]:
# Insert data into recommender_output_priority (overwrite behavior)
spark.sql(f"""
INSERT OVERWRITE TABLE revr_bmbs_dmp_offline.recommender_output_priority
WITH loyalty_vins AS (
    SELECT DISTINCT oob.vin
    FROM datalake_tireo2o.dw_cust_tireo2o_customer_right_coupon crc
    INNER JOIN datalake_tireo2o.dw_cust_tireo2o_customer_loyalty_plan clp ON clp.id = crc.customer_loyalty_plan_id
    INNER JOIN datalake_tireo2o.dw_cust_tireo2o_order_order_base oob ON clp.order_id = oob.id
    WHERE oob.order_type_flag = 1 AND crc.coupon_type = 1
),
tire_package_vins AS (
    SELECT DISTINCT oob.vin
    FROM datalake_tireo2o.dw_cust_tireo2o_customer_right_coupon crc
    INNER JOIN datalake_tireo2o.dw_cust_tireo2o_customer_loyalty_plan clp ON clp.id = crc.customer_loyalty_plan_id
    INNER JOIN datalake_tireo2o.dw_cust_tireo2o_order_order_base oob ON clp.order_id = oob.id
    WHERE oob.order_type_flag = 1 AND crc.coupon_type = 4
),
maintenance_vins AS (
    SELECT DISTINCT b.vin
    FROM datalake_tireo2o.dw_cust_tireo2o_customer_right_coupon a
    INNER JOIN datalake_tireo2o.dw_cust_tireo2o_order_order_base b ON a.order_id = b.id
    WHERE a.status_flag IN (0, 1, 2, 6, 12, 13)
),
ew_vins AS (
    SELECT DISTINCT eoc.vin
    FROM datalake_tireo2o.dw_srvc_tireo2o_ew_order_coupon oc
    INNER JOIN datalake_tireo2o.dw_dealer_tireo2o_ew_order eo ON eo.id = oc.order_id
    INNER JOIN datalake_tireo2o.dw_vhcl_tireo2o_ew_order_car eoc ON eo.id = eoc.order_id
    WHERE oc.status IN ('CREATED', 'PENDING_ACTIVATION', 'PENDING_EFFECT', 'EFFECTIVE', 'EFFECTIVE_CLAIMED', 'PENDING_APPROVAL')
),
mmc_products AS (
    SELECT 
    p.code AS product_id,
    CASE 
        WHEN p.product_type = 1 THEN 'MB'
        WHEN p.product_type = 2 AND (p.third_party_membership IS NULL OR p.third_party_membership = FALSE) THEN 'UNICOM'
        WHEN p.product_type = 6 THEN 'PACKAGE'
        ELSE NULL 
    END AS product_type
FROM 
    datalake_ecommerce.dw_srvc_ecommerce_mmc_variant_product vp 
JOIN 
    datalake_ecommerce.dw_srvc_ecommerce_mmc_product p
ON 
    vp.product_code = p.code
WHERE 
    (p.product_type IN (1, 6) 
    OR (p.product_type = 2 AND (p.third_party_membership IS NULL OR p.third_party_membership = FALSE)))
    AND DATE_DIFF(DAY, TO_DATE(vp.created_time), CURRENT_DATE) <= 30
),
mmc_coupon_products AS (   -- NEW: MMC products with valid coupon (your Delta table)
    SELECT
        UPPER(product_id)   AS product_id,
        UPPER(product_type) AS product_type
    FROM revr_bmbs_dmp_offline.mmc_valid_coupon_products
)

SELECT
    r.vin,
    r.product_id,
    r.product_type,
    r.product_category,
    r.score,
    CASE WHEN r.vin IN (SELECT vin FROM loyalty_vins) AND r.product_type = 'LOYALTY' THEN 1 ELSE 0 END AS loyalty_flag,
    CASE WHEN r.vin IN (SELECT vin FROM tire_package_vins) AND r.product_type = 'TIREPACKAGE' THEN 1 ELSE 0 END AS tire_package_flag,
    CASE WHEN r.vin IN (SELECT vin FROM maintenance_vins) AND r.product_type = 'SCPRODUCT' THEN 1 ELSE 0 END AS maintenance_flag,
    CASE WHEN r.vin IN (SELECT vin FROM ew_vins) AND r.product_type = 'EW' THEN 1 ELSE 0 END AS ew_flag,
    CASE
        WHEN
            (r.vin IN (SELECT vin FROM loyalty_vins) AND r.product_type = 'LOYALTY') OR
            (r.vin IN (SELECT vin FROM tire_package_vins) AND r.product_type = 'TIREPACKAGE') OR
            (r.vin IN (SELECT vin FROM maintenance_vins) AND r.product_type = 'SCPRODUCT') OR
            (r.vin IN (SELECT vin FROM ew_vins) AND r.product_type = 'EW')
        THEN 1 ELSE 0
    END AS lpo_flag,

    CASE  -- UPDATED: keep old mmc_flag behavior, but make comparison case-insensitive
        WHEN r.product_category = 'MMC' AND (UPPER(r.product_id), UPPER(r.product_type)) IN (SELECT UPPER(product_id), UPPER(product_type) FROM mmc_products)
        THEN 1 ELSE 0
    END AS mmc_flag,

    CASE  -- NEW: has valid coupon for this MMC product
        WHEN r.product_category = 'MMC' AND (UPPER(r.product_id), UPPER(r.product_type)) IN (SELECT product_id, product_type FROM mmc_coupon_products)
        THEN 1 ELSE 0
    END AS mmc_coupon_flag,

    CASE  -- NEW: MMC priority rank (1 is highest)
        WHEN r.product_category = 'MMC' AND UPPER(r.product_type) = 'PACKAGE'
             AND (UPPER(r.product_id), UPPER(r.product_type)) IN (SELECT product_id, product_type FROM mmc_coupon_products)
        THEN 1
        WHEN r.product_category = 'MMC' AND UPPER(r.product_type) = 'PACKAGE'
        THEN 2
        WHEN r.product_category = 'MMC' AND UPPER(r.product_type) <> 'PACKAGE'
             AND (UPPER(r.product_id), UPPER(r.product_type)) IN (SELECT product_id, product_type FROM mmc_coupon_products)
        THEN 3
        WHEN r.product_category = 'MMC' AND (UPPER(r.product_id), UPPER(r.product_type)) IN (SELECT UPPER(product_id), UPPER(product_type) FROM mmc_products)
        THEN 4
        WHEN r.product_category = 'MMC'
        THEN 5
        ELSE NULL
    END AS mmc_priority_rank,

    CASE  -- UPDATED: adjusted_score now supports multiple MMC priorities (and keeps LPO boost)
        WHEN
            ((r.vin IN (SELECT vin FROM loyalty_vins) AND r.product_type = 'LOYALTY') OR
            (r.vin IN (SELECT vin FROM tire_package_vins) AND r.product_type = 'TIREPACKAGE') OR
            (r.vin IN (SELECT vin FROM maintenance_vins) AND r.product_type = 'SCPRODUCT') OR
            (r.vin IN (SELECT vin FROM ew_vins) AND r.product_type = 'EW'))
        THEN r.score + 100

        WHEN r.product_category = 'MMC' AND UPPER(r.product_type) = 'PACKAGE'
             AND (UPPER(r.product_id), UPPER(r.product_type)) IN (SELECT product_id, product_type FROM mmc_coupon_products)
        THEN r.score + 50   -- Priority 1: Package product with coupon

        WHEN r.product_category = 'MMC' AND UPPER(r.product_type) = 'PACKAGE'
        THEN r.score + 40   -- Priority 2: Package product

        WHEN r.product_category = 'MMC' AND UPPER(r.product_type) <> 'PACKAGE'
             AND (UPPER(r.product_id), UPPER(r.product_type)) IN (SELECT product_id, product_type FROM mmc_coupon_products)
        THEN r.score + 30   -- Priority 3: Normal product with coupon

        WHEN r.product_category = 'MMC' AND (UPPER(r.product_id), UPPER(r.product_type)) IN (SELECT UPPER(product_id), UPPER(product_type) FROM mmc_products)
        THEN r.score + 20   -- Priority 4: New product (your mmc_products CTE is last 30 days)

        WHEN r.product_category = 'MMC'
        THEN r.score + 10   -- Priority 5: Normal product

        ELSE r.score
    END AS adjusted_score,
    '{dt_value}' AS dt
FROM revr_bmbs_dmp_offline.recommender_output r
WHERE dt = '{dt_value}'
""")


In [0]:
# Create a partitioned table by 'dt'
spark.sql(f"""
CREATE TABLE IF NOT EXISTS revr_bmbs_dmp_offline.recommender_output_priority_partitioned (
    vin STRING,
    product_id STRING,
    product_type STRING,
    product_category STRING,
    score DOUBLE,
    loyalty_flag INT,
    tire_package_flag INT,
    maintenance_flag INT,
    ew_flag INT,
    lpo_flag INT,
    mmc_flag INT,
    adjusted_score DOUBLE,
    mmc_coupon_flag   INT,
    mmc_priority_rank INT
)
PARTITIONED BY (dt STRING)
""")



In [0]:

# Insert daily data into the partitioned table (overwrite specific partition)
spark.sql(f"""
INSERT OVERWRITE TABLE revr_bmbs_dmp_offline.recommender_output_priority_partitioned PARTITION (dt = '{dt_value}')
SELECT vin,product_id,product_type,product_category,score,loyalty_flag,tire_package_flag,maintenance_flag,ew_flag,lpo_flag,mmc_flag,adjusted_score,mmc_coupon_flag,mmc_priority_rank FROM revr_bmbs_dmp_offline.recommender_output_priority
WHERE dt = '{dt_value}'
""")